# Introduction to AI powered Services Part 2

In v2.4.3 of the ArCGIS API for Python, we released support for three AI powered services in ArcGIS Online via the [`arcgis.ai`](https://developers.arcgis.com/python/latest/api-reference/arcgis.ai.html) module. To learn more about those tools, please refer to [this guide](https://developers.arcgis.com/python/latest/guide/introduction-to-ai-powered-services/). 

With __v2.4.4 of the ArcGIS API for Python__, we released support for the following three new AI powered services. Another major enhancement introduced with this version of the API and with __v12.2 of ArcGIS Enterprise__ is the ability to run these AI powered services in ArcGIS Enterprise.

1. [`analyze_file()`](https://developers.arcgis.com/python/latest/api-reference/arcgis.ai.html#analyze-file) - Analyzes files based on user provided instructions.
2. [`transcribe_audio()`](https://developers.arcgis.com/python/latest/api-reference/arcgis.ai.html#transcribe-audio) - Transcribe audio files into text using machine learning models.
3. [`generate_speech()`](https://developers.arcgis.com/python/latest/api-reference/arcgis.ai.html#generate-speech) - Converts input text to synthesized speech using Azure AI speech.


> Note:
> An ArcGIS Online organizational account and internet access are required to enable and use AI capabilities in ArcGIS Enterprise. The account you specify when enabling the capabilities must be of a supported ArcGIS Online organizational account type. The capabilities must be enabled in ArcGIS Online before you can enable them in ArcGIS Enterprise.
Data and prompts used in conjunction with AI capabilities will be processed in ArcGIS Online. Any data used in conjunction with an AI capability can be sent to this external service managed by Esri

To enable this capability for your organization, click _Enable_, provide the credentials for an ArcGIS Online organizational account, and click _Accept and enable_. If the capability is already enabled and you need to update the credentials, click _Update credentials_, provide the new credentials, and click _Update ArcGIS Online credentials_.

In [2]:
from arcgis.gis import GIS
from arcgis.ai import image_extract_text, analyze_file, transcribe_audio, generate_speech, AIUtilsResponse

In [18]:
gis = GIS(profile="your_ent_admin_profile")

### __Analyze File__

This tool analyzes files based on user provided instructions for several use-cases such as summarizing the file, extracting text, classifying contents of the file, and performing workflows that assist with understanding the file. You can analyze PDF files using this tool by providing inputs as the URL of the file hosted on *.arcgis.com domains OR as Base64 encoded file content.

Let us take a look at an example to analyze this 37 page document for [District of Columbia Chief Data Officer's Annual Report in 2020](https://geosaurus.maps.arcgis.com/home/item.html?id=a8ca5af1fa34467699bae7d1d4d688e6#overview).

![dc_report](https://github.com/user-attachments/assets/84d5c529-9739-4cf5-b88a-3dd1e3d3149f)

In [5]:
file_url = "https://geosaurus.maps.arcgis.com/sharing/rest/content/items/a8ca5af1fa34467699bae7d1d4d688e6/data"

Prompts to the `analyze_file()` method may be provided as string, list or dictionary inputs. We will see examples for all 3 below. 

#### 1. Prompt as `str` input

In [4]:
prompt_str = "Summarize the contents of the file."

In [26]:
summary = analyze_file(file = file_url, prompt = prompt_str)
type(summary)

arcgis.ai._utils.AIUtilsResponse

As we will continue to see with the tools within the `arcgis.ai` module, the results generated from running these tools in the ArcGIS API for Python are of [`AIUtilsResponse`](https://developers.arcgis.com/python/latest/api-reference/arcgis.ai.html#arcgis.ai.AIUtilsResponse) type. 

You can verify if the request was successful using the `success` field of the response. 

summary.success

Since `AIUtilsResponse` is a pydantic model, you can access the specific results of the response as shown below.

In [25]:
summary.results[0].value

'The file is the District of Columbia Office of the Chief Technology Officer’s 2020 Chief Data Officer Annual Report. It reviews the District’s data policy, current data management status, 2020 Enterprise Dataset Inventory (EDI) results, FOIA tracking, 2019 accomplishments, and future goals aligned with OCTO’s draft technology strategic plan.\n\nKey points:\n- The report is framed by DC’s data policy and OCTO’s draft strategic plan, especially Commitment 4: managing, organizing, and leveraging data to support government mission delivery.\n- The 2020 EDI recorded 1,915 enterprise datasets from 82 agencies, up from 1,779 datasets from 75 agencies in 2019.\n- Dataset classifications in 2020 included 859 Open (Level 0), 179 Public but Not Proactively Released (Level 1), 219 For District Government Use (Level 2), 562 Confidential (Level 3), and 97 Restricted Confidential (Level 4).\n- Of 873 Level 0 datasets, 628 were available on the Open Data Portal; 245 open datasets were still not poste

The tool gives us a succinct and clear summary of the document.

We will now proceed to see examples involving the other input types through this item

#### 2. Prompt as `list` input

In [31]:
prompt_list = [
    "Describe the accomplishments from 2019 with regards to the Environment.",
    "What are the categories of data available?"
]

In [32]:
result_list = analyze_file(file = file_url, prompt = prompt_list)
result_list

AIUtilsResponse(success=True, meta=AIUtilsResponseMetaData(usage=None, model='gpt-5.4'), results=[AnalyzeResult(success=True, value='For the Environment, the 2019 accomplishment highlighted was OCTO’s use of data storytelling tools. The report states that environment data stories were the most popular. A featured example was the Department of General Services’ story, “The Story of a Banana Peel.” It lets students trace a banana from a farm in Ecuador through the Port of Philadelphia to a DC Public School cafeteria, and then follow how the peel is composted and becomes fertile dirt used at Nationals Park and elsewhere. The report also notes that there are many such environmental stories on the Open Data Portal.', key='unique_string_0'), AnalyzeResult(success=True, value=['Government Operations', 'Transportation', 'Education', 'Health and Human Services', 'Public Services', 'Public Safety', 'Environment', 'Business and Economic Development', 'Administrative and Other Boundaries', 'Utilit

We can access results for the individual prompts via the prompt index provided to the `results` field as demonstrated below.

In [33]:
result_list.results[0].value

'For the Environment, the 2019 accomplishment highlighted was OCTO’s use of data storytelling tools. The report states that environment data stories were the most popular. A featured example was the Department of General Services’ story, “The Story of a Banana Peel.” It lets students trace a banana from a farm in Ecuador through the Port of Philadelphia to a DC Public School cafeteria, and then follow how the peel is composted and becomes fertile dirt used at Nationals Park and elsewhere. The report also notes that there are many such environmental stories on the Open Data Portal.'

In [34]:
result_list.results[1].value

['Government Operations',
 'Transportation',
 'Education',
 'Health and Human Services',
 'Public Services',
 'Public Safety',
 'Environment',
 'Business and Economic Development',
 'Administrative and Other Boundaries',
 'Utility and Communication',
 'Property and Land',
 'Financial',
 'Planning Land Use and Zoning',
 'Aerial Photography and Scanned Maps',
 'Historic',
 'Demographic',
 'Cultural and Society',
 'Facility and Structure',
 'Recreation',
 'Location',
 'Technology',
 'Elevation',
 'Communication',
 'Basemap']

The response gives us concise and complete responses for our prompts.

#### 3. Prompt as `dict` input 

In order to provide prompts as a dictionary, it is essential to provide the values for the `key` (unique values) and `context` keys of the dictionary for each prompt that you wish to provide. 

In [35]:
prompt_dicts = [
    {"key": "accomplishments", "context": "Describe the accomplishments from 2019 with regards to Mappign and Analysis"},
    {"key": "takeaways", "context": "What takeaways from the report are they taking into the following year?"}
]

In [37]:
result_dicts = analyze_file(file = file_url, prompt = prompt_dicts)
result_dicts

AIUtilsResponse(success=True, meta=AIUtilsResponseMetaData(usage=None, model='gpt-5.4'), results=[AnalyzeResult(success=True, value={'technical_capability': 'Mapping and Analysis', 'summary': 'In 2019, Esri Inc. recognized the District of Columbia Government for creating a leading model for governments and businesses through its use of GIS. The report says Esri cited the District’s GIS work in creating greater efficiency, collaboration across departments, transparent and open government, and citizen engagement. It also noted the central GIS/IT infrastructure provided by OCTO to all agencies and the 1,200 datasets in its open data portal used to share information across departments.', 'section': 'Esri Enterprise Geographic Information System (GIS) Award'}, key='accomplishments'), AnalyzeResult(success=True, value=['The report says future work will be guided by the Draft Technology Strategic Plan for DC: Unleashing the Possible, especially Commitment 4: manage, organize, and leverage dat

In [52]:
result_dicts.results[0].value

{'technical_capability': 'Mapping and Analysis',
 'summary': 'In 2019, Esri Inc. recognized the District of Columbia Government for creating a leading model for governments and businesses through its use of GIS. The report says Esri cited the District’s GIS work in creating greater efficiency, collaboration across departments, transparent and open government, and citizen engagement. It also noted the central GIS/IT infrastructure provided by OCTO to all agencies and the 1,200 datasets in its open data portal used to share information across departments.',
 'section': 'Esri Enterprise Geographic Information System (GIS) Award'}

In [40]:
result_dicts.results[1].value

['The report says future work will be guided by the Draft Technology Strategic Plan for DC: Unleashing the Possible, especially Commitment 4: manage, organize, and leverage data to facilitate DC government mission delivery.',
 'They plan to identify high-value mission use cases tied to Mayoral priorities, including stronger support for housing and a deeper focus on one DC government cluster in the next year.',
 'They intend to continue streamlining data sharing through a multilateral data-sharing agreement template and automation of the eMOU process.',
 'They will keep working to move more Level 0 datasets onto the Open Data Portal and make the Enterprise Dataset Inventory more interactive and useful.',
 'They plan to continue improving FOIAXpress adoption, training, and public access, while using FOIA data to identify datasets that should be opened.',
 'They want to develop technology and processes to help agencies anonymize sensitive data so it can be shared and analyzed more safely.

The tool here correctly identifies the successes in the field of 'Mapping and Analysis' and also summarizes the key takeaways aptly. 

### __Trascribe Audio__

This tool converts audio files into written text using machine learning models. You can analyze audio by providing inputs as the URL of the audio file hosted on *.arcgis.com domains OR as Base64 encoded payload. Currently supported file formats are: .wav, .mp3, .m4a, .mp4, .aac, .ogg, .opus, .webm, .flac, .amr, .amr-wb.

We will look at an example to transcribe audio from a an audio file on my local machine converted to Base64 encoded audio payload. Please replace the path to the audio file below to an audio file on your machine.

In [63]:
audio_file = "D:/Github Projects/geosaurus/tests/resources/storymap/rocks.mp3"

In order to listen to audio file from the notebook experience, you can embed your audio file as shown below:

In [64]:
from IPython.display import Audio

Audio(filename=audio_file, embed=True)

We will now write a method that converts this file to base64 string that can be provided for transcribing to our AI tool.

In [44]:
import base64

def audio_to_base64(file_path):
    with open(file_path, "rb") as audio_file:
        # Read the binary data from the file
        binary_data = audio_file.read()
        # Encode the binary data to Base64 bytes
        base64_bytes = base64.b64encode(binary_data)
        # Decode to a UTF-8 string for JSON serialization
        base64_string = base64_bytes.decode("utf-8")
    return base64_string

In [48]:
encoded_audio = audio_to_base64(audio_file)

We will now use the `transcribe_audio()` method with this `encoded_audio` input. The method also takes a `from_language` optional input parameter with "en" (English) as default. To identify the languages supported in your ArcGIS Online Organization and the culture of those languages, you may invoke the `languages` property of the `GIS` object.

In [51]:
transcribed_text = transcribe_audio(audio = f"data:audio/mp4;base64,{encoded_audio}")
transcribed_text.results[0].value

'This large exposed rock was a favorite location for Carl Sandburg to bring a chair and orange crate for an outdoor study. He enjoyed the quiet and peacefulness of this location to edit his work. Exposed rocks like this are located throughout the area and have a long history of use, going back to the first peoples who settled the area we know today as Flat Rock.'

This pattern demonstrated to convert local files to base64 string inputs can be applied to files on your local machine for the `analyze_file()` method shown above as well.

### __Generate Speech__

This tool performs the inverse operation of the Transcribe Audio tool, i.e., it converts input text to synthesized speech using Azure AI speech. The response is streamed audio. The default output format is MP3. Similar to the Transcribe Audio tool, this method also takes a `from_language` optional input parameter with "en" (English) as default.

Let's look at an example that converts the following text to speech.

In [55]:
text_input = "The ArcGIS API for Python is a powerful, modern Pythonic library that supports the latest releases of ArcGIS Enterprise and \
            ArcGIS Online and provides a consistent programmatic experience for scripting and automating across the ArcGIS product suite. It is \
            used for three key workflows involving GIS organization administration, Content management and Spatial analysis and data science."

In [57]:
speech = generate_speech(text = text_input)

Since the result contains synthesized audio as a string binary payload, we need to decode it to access it.

In [58]:
# Decode the Base64 string into binary data
audio_bytes = base64.b64decode(speech.results[0].value)

# Write the binary data to an audio file (Supported file formats - `mp3`, `raw`, `ogg`, and `webm`.)
with open("generate_speech_audio.wav", "wb") as audio_file:
    audio_file.write(audio_bytes)

Let us embed the output file in the notebook and verify the text to speech conversion.

In [61]:
from IPython.display import Audio

Audio(filename="generate_speech_audio.wav", embed=True)

### Conclusion

The `arcgis.ai` module is useful when your workflows involve unstructured data in the form of documents covering information including but not limited to planning, assessments, studies, reports, etc. The audio tools can prove beneficial when your workflows involve field recordings, voice notes, inspection reports, instructions, advisories, information and isntructions. Deriving necessary information from these data sources using these services supported within the ArcGIS API for Python can improve productivity. 

Below are a few related resources to further your understanding and familiarity with this concept:
1. [Introduction to AI Powered services (Part 1)](https://developers.arcgis.com/python/latest/guide/introduction-to-ai-powered-services/)
2. [Enhancing 311 Service Requests with ArcGIS AI Utility Services](https://developers.arcgis.com/python/latest/samples/ai-utility-services-enhancing-service-requests/)